# Resultados: pré-processamento, grades e auditoria

Rode `bash scripts/run_overnight.sh` antes. Este notebook só mostra os resultados.

**Dados restritos:** as saídas contêm imagens de pacientes. Não compartilhe o arquivo com saídas; o filtro do Git as remove nos commits.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd() if (Path.cwd() / 'hyamd').exists() else Path.cwd().parent
os.chdir(ROOT)
CACHE = Path(os.environ.get('MACDEG_CACHE', 'cache'))
AUDIT = Path(os.environ.get('MACDEG_AUDIT', 'outputs/audit'))
GRIDS = Path(os.environ.get('MACDEG_GRIDS', 'outputs/grids'))
pd.set_option('display.max_rows', 300, 'display.max_columns', 40)

def show(path, width=15):
    image = Image.open(path)
    plt.figure(figsize=(width, width * image.height / image.width))
    plt.imshow(np.asarray(image))
    plt.axis('off')
    plt.title(Path(path).name, loc='left')
    plt.show()

## Sinalizações

Alertas acima de 10% e quais ROIs aceitam rotação. Compare as taxas entre classes: um detector pior nos olhos com DMRI enviesaria os recortes.

In [ ]:
summary = json.loads((CACHE / 'flag_summary.json').read_text())
print('Alertas:', *summary['alerts'], sep='\n  ')
print('Rotação permitida:', summary['rotation_allowed'])
print('Cache em disco: %.1f GB' % (summary['bytes_on_disk'] / 1e9))
pd.DataFrame({roi: {k: v for k, v in rates.items() if k != 'by_AMD'} for roi, rates in summary['by_roi'].items()}).T

In [ ]:
pd.DataFrame({(roi, amd): flags for roi, rates in summary['by_roi'].items() for amd, flags in rates['by_AMD'].items()}).T

## Localização e recortes

Caixa verde: ROI. Círculo magenta: fóvea. Círculo ciano: disco. Cruz branca: centro do campo.

**Verifique:** o ciano ainda cai em drusas? O magenta está sobre a fóvea? Nas imagens com disco incerto, a caixa continua em torno da mácula?

In [ ]:
for path in sorted(GRIDS.glob('localizacao_*.png')) + sorted(GRIDS.glob('caixa_recorte_*.png')):
    show(path)

## Níveis de augmentation

A3 é controle negativo: deve parecer visivelmente degradado.

In [ ]:
for path in sorted(GRIDS.glob('nivel_*.png')):
    show(path)

## Auditoria

Proxy comparativo: compare variantes entre si, sem ler os valores como taxa de drusas preservadas. O valor de `A0` é a linha de base de cada ROI e resolução.

In [ ]:
verdict = json.loads((AUDIT / 'audit_verdict.json').read_text())
print(json.dumps(verdict, indent=1))
pd.read_csv(AUDIT / 'negative_controls.csv').round(3)

In [ ]:
table = pd.read_csv(AUDIT / 'audit_table.csv')
columns = ['variant', 'n_images', 'px_per_dd_mean', 'retention_mean', 'retention_p5', 'retention_p95', 'retention_low_contrast_mean', 'contrast_ratio_mean', 'energy_ratio_mean', 'spurious_rate_mean', 'roi_area_lost_mean']
table[columns].round(3)

In [ ]:
for path in sorted(AUDIT.glob('*.png')):
    show(path, width=14)